In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append("..")

In [ ]:
import torch
import torch.fft
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

import numpy as np

import time

import random

import soap
from architectures import PINO1D
from generators import SlabWaveguideDataset
from utils import analytical_solution, solve_fd_mode

In [ ]:
torch.manual_seed(42);

In [ ]:
device = 'cuda'

In [ ]:
dataset = SlabWaveguideDataset(dx = 1/20, stochastic=True, n_dataset=64, x_range=(-5.0, 5.0), device=device)
loader = DataLoader(dataset, batch_size=64, shuffle=False)

val_ds = SlabWaveguideDataset(dx = 1/20, stochastic=True, n_dataset=1024, x_range=(-5.0, 5.0), device=device)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

In [ ]:
# Setup
model = PINO1D(4, 1, 64, 32, 4).to(device)

In [ ]:
checkpoint_no = np.load('model_checkpoints/last_checkpoint.npy').item()
checkpoint = torch.load(f"model_checkpoints/slab_full_checkpoint_{checkpoint_no}.pth", weights_only=False)

epoch = checkpoint["epoch"]
model.load_state_dict(checkpoint["model_state_dict"])

train_loss_history = checkpoint["train_loss_history"]
val_loss_history = checkpoint["val_loss_history"]
betas_avg = checkpoint["betas_avg"]
analytical_EIs = checkpoint['analytical_EIs.npy'].tolist()

print('loaded latest parameteres')

In [ ]:
def evaluate_model_errors(model, dataset, device):
    field_errors_pino = []
    field_errors_fd = []
    eigenvalue_errors_pino = []
    eigenvalue_errors_fd = []

    model.eval()
    for idx in range(len(dataset)):
        xyr, features = dataset[idx]
        xyr = xyr.unsqueeze(0).to(device)
        features = features.unsqueeze(0).to(device)
        dx = dataset.dx

        k0 = features[0, 0].item()
        TE = features[0, -1].item()
        n2_max = features[0, -2]
        n2_min = torch.minimum(features[0, -4], features[0, -3])

        with torch.no_grad():
            E_out, beta_nn = model(xyr)
        E_out = E_out.squeeze().cpu().numpy()
        x_out = xyr[:, 0, :].squeeze().cpu().numpy()
        ri_out = xyr[:, 1, :].squeeze().cpu().numpy()

        beta_sq = (n2_max - beta_nn * (n2_max - n2_min)) * k0**2
        beta_pino = torch.sqrt(beta_sq).item()

        # Solve FD
        beta_fd, modes_fd = solve_fd_mode(ri_out, dx, k0, TM=(TE == 0), num_modes=1)
        mode_fd = modes_fd[0]

        # Analytical
        n_clad = torch.sqrt(features[0, -4]).item()
        n_subs = torch.sqrt(features[0, -3]).item()
        n_core = torch.sqrt(features[0, -2]).item()
        w_core = features[0, 1].item()
        analytical_modes, analytical_EIs = analytical_solution(x_out, n_core, n_subs, n_clad, w_core, k0, TE==0)
        mode_ana = analytical_modes[0]
        beta_ana = analytical_EIs[0] * k0

        # Normalize fields
        E_out /= np.linalg.norm(E_out)
        mode_fd /= np.linalg.norm(mode_fd)
        if TE == 0:
            mode_ana = mode_ana / ri_out
        mode_ana /= np.linalg.norm(mode_ana)

        # Fix signs
        sign_pino = np.sign(np.dot(E_out, mode_ana))
        sign_fd = np.sign(np.dot(mode_fd, mode_ana))
        E_out *= sign_pino
        mode_fd *= sign_fd

        # Errors
        err_pino = 100 * np.linalg.norm(E_out - mode_ana) / np.linalg.norm(mode_ana)
        err_fd = 100 * np.linalg.norm(mode_fd - mode_ana) / np.linalg.norm(mode_ana)
        
        field_errors_pino.append(err_pino)
        field_errors_fd.append(err_fd)

        eigenvalue_errors_pino.append(np.abs((beta_pino - beta_ana) / beta_ana))
        eigenvalue_errors_fd.append(np.abs((beta_fd - beta_ana) / beta_ana))

    return {
        "field_error_pino": np.mean(field_errors_pino),
        "field_error_fd": np.mean(field_errors_fd),
        "eigen_error_pino": np.mean(eigenvalue_errors_pino),
        "eigen_error_fd": np.mean(eigenvalue_errors_fd),
    }

In [ ]:
start_time = time.time()
errors = evaluate_model_errors(model, val_ds, device)
end_time = time.time()
print('this took', end_time - start_time, 'seconds')
print("PINO Field Error (relative L2):", errors["field_error_pino"])
print("FD Field Error (relative L2):", errors["field_error_fd"])
print("PINO Eigenvalue Error (avg rel):", errors["eigen_error_pino"])
print("FD Eigenvalue Error (avg rel):", errors["eigen_error_fd"])

In [ ]:
def evaluate_param_sweep(model, sweep_results, device):
    eigenvalues_pino = []
    eigenvalues_analytic = []
    eigenvalues_fd = []

    model.eval()
    for idx in range(len(sweep_results)):
        xyr, features = sweep_results[idx]
        xyr = xyr.unsqueeze(0).to(device)
        features = features.unsqueeze(0).to(device)
        dx = dataset.dx
        ri_out = xyr[:, 1, :].squeeze().cpu().numpy()

        k0 = features[0, 0].item()
        TE = features[0, -1].item()
        n2_max = features[0, -2]
        n2_min = torch.minimum(features[0, -4], features[0, -3])

        with torch.no_grad():
            E_out, beta_nn = model(xyr)
        beta_sq = (n2_max - beta_nn * (n2_max - n2_min))
        beta_pino = torch.sqrt(beta_sq).item()
        
        beta_fd, modes_fd = solve_fd_mode(ri_out, dx, k0, TM=(TE == 0), num_modes=1)
        
        # Analytical
        x_out = xyr[:, 0, :].squeeze().cpu().numpy()
        n_clad = torch.sqrt(features[0, -4]).item()
        n_subs = torch.sqrt(features[0, -3]).item()
        n_core = torch.sqrt(features[0, -2]).item()
        w_core = features[0, 1].item()
        analytical_modes, analytical_EIs = analytical_solution(x_out, n_core, n_subs, n_clad, w_core, k0, TE==0)
        beta_ana = analytical_EIs[0]

        eigenvalues_pino.append(beta_pino)
        eigenvalues_analytic.append(beta_ana)
        eigenvalues_fd.append(beta_fd/k0)

    return {
        "eigenvalues_pino": eigenvalues_pino,
        "eigenvalues_analytic": eigenvalues_analytic,
        "eigenvalues_fd": eigenvalues_fd,
    }


In [ ]:
def build_grid_aligned_values(vmin, vmax, dx, mode="center"):
    """
    Generate values that are exactly representable on the grid.
    """
    if mode == "center":
        k_min = int(np.ceil(vmin / dx - 0.5))
        k_max = int(np.floor(vmax / dx - 0.5))
        values = (np.arange(k_min, k_max + 1) + 0.5) * dx

    elif mode == "edge":
        k_min = int(np.ceil(vmin / dx))
        k_max = int(np.floor(vmax / dx))
        values = (np.arange(k_min, k_max + 1) + 1e-2) * dx

    else:
        raise ValueError

    return values

In [ ]:
sweep_results_list = []
for dx in [1/20, 1/40]:
    dataset = SlabWaveguideDataset(dx = dx, stochastic=True, n_dataset=32, x_range=(-5, 5), device=device)
    
    x_range = 2 + np.linspace(0.05, 1.95, 25)
    param_dict_list = [
        {"width": 1.5, "n_clad": 1.44, "n_sub": 1.44, "n_core": x, "TE": True,  "wavelength": 1.55} for x in x_range
    ]
    
    sweep_samples = dataset.parameter_sweep(param_dict_list)
    sweep_results = evaluate_param_sweep(model, sweep_samples, device)
    sweep_results_list.append(sweep_results)

In [ ]:
ranges =[0, len(x_range)//2, len(x_range)]
markers = ['o', '*']
plt.figure(figsize=(8,5))
plot_colors = ['#FF69B4', 'black']
# PINO points (scatter)
# Analytic curve (line)
plt.plot(
    x_range, sweep_results['eigenvalues_analytic'],
    label='Analytic',
    color='#1E90FF',   # stronger blue (dodger blue)
    linewidth=2.5,
    alpha=0.9
)

for i in range(2):
# PINO points (scatter)
    plt.scatter(
        x_range, sweep_results_list[i]['eigenvalues_pino'],
        label=f'PINO (dx=1/{i*20 + 20})',
        marker='o',
        s=80,
        color=plot_colors[i],   # hot pink
        facecolors='none',        # hollow center
        edgecolors=plot_colors[i],     # pink edge
        linewidth=0.8,
        alpha=0.9,
        zorder=2
    )

plt.axvline(x=3.5, color='tab:blue', linewidth=3, linestyle='dashed')
plt.annotate('in-distribution', (3.1, 2), fontsize=12)
plt.annotate('out-of-distribution', (3.55, 2), fontsize=12)

# Style
plt.xlim([2, 4])
plt.xlabel("$n_{co}$", fontsize=16)
plt.ylabel("$n_{eff}$", fontsize=16)
plt.legend(frameon=False, fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
sweep_results_list = []
for dx in [1/25, 1/50]:
    dataset = SlabWaveguideDataset(dx = dx, stochastic=True, n_dataset=32, x_range=(-5, 5), device=device)
    
    x_range = 1 + np.linspace(0.05, 2.35, 25)
    
    param_dict_list = [
        {"width": 1.5, "n_clad": 1.44, "n_sub": x, "n_core": 3.47, "TE": True,  "wavelength": 1.55} for x in x_range
    ]
    
    sweep_samples = dataset.parameter_sweep(param_dict_list)
    sweep_results = evaluate_param_sweep(model, sweep_samples, device)
    sweep_results_list.append(sweep_results)

In [ ]:
plt.figure(figsize=(8,5))

plot_colors = ['#FF69B4', 'black']

# Analytic curve (line)
plt.plot(
    x_range, sweep_results['eigenvalues_analytic'],
    label='Analytic',
    color='#1E90FF',   # stronger blue (dodger blue)
    linewidth=2.5,
    alpha=0.9
)

# PINO points (scatter)
for i in range(2):
# PINO points (scatter)
    plt.scatter(
        x_range, sweep_results_list[i]['eigenvalues_fd'],
        label=f'PINO (dx=1/{i*25 + 25})',
        marker='o',
        s=80,
        color=plot_colors[i],   # hot pink
        facecolors='none',        # hollow center
        edgecolors=plot_colors[i],     # pink edge
        linewidth=0.8,
        alpha=0.9,
        zorder=2
    )

plt.axvline(x=2.0, color='tab:blue', linewidth=3, linestyle='dashed')
plt.annotate('in-distribution', (1.25, 3.442), fontsize=12)
plt.annotate('out-of-distribution', (2.25, 3.442), fontsize=12)

# Style
plt.xlabel("$n_{sub}$", fontsize=16)
plt.xlim([1, 3.5])
plt.ylabel("$n_{eff}$", fontsize=16)
plt.legend(frameon=False, fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
sweep_results_list = []

for dx in [1/20, 1/40]:
    dataset = SlabWaveguideDataset(dx = dx, stochastic=True, n_dataset=32, x_range=(-5, 5), device=device)
    
    x = build_grid_aligned_values(1.05, 2.95, 1/10, mode="edge")
    
    param_dict_list = [
        {"width": val.item(), "n_clad": 1.44, "n_sub": 1.05, "n_core": 3.47, "TE": True,  "wavelength": 1.55} for val in x
    ]
    
    sweep_samples = dataset.parameter_sweep(param_dict_list)
    sweep_results = evaluate_param_sweep(model, sweep_samples, device)
    sweep_results_list.append(sweep_results)

In [ ]:
plt.figure(figsize=(8,5))
plot_colors = ['#FF69B4', 'black']

# PINO points (scatter)
for i in range(2):
    plt.scatter(
        x, sweep_results_list[i]['eigenvalues_fd'],
        label=f'PINO (dx=1/{i*20 + 20})',
        marker='o',
        s=80,
        color=plot_colors[i],   # hot pink
        facecolors='none',        # hollow center
        edgecolors=plot_colors[i],     # pink edge
        linewidth=0.8,
        alpha=0.9
    )

# Analytic curve (line)
plt.plot(
    x, sweep_results['eigenvalues_analytic'],
    label='Analytic',
    color='#1E90FF',   # stronger blue (dodger blue)
    linewidth=2.5,
    alpha=0.9
)

plt.axvline(x=2.0, color='tab:blue', linewidth=3, linestyle='dashed')
plt.annotate('in-distribution', (1.1, 3.45), fontsize=12)
plt.annotate('out-of-distribution', (2.25, 3.45), fontsize=12)


# Style
plt.xlabel("$w_{co}$", fontsize=16)
plt.xlim([1, 3])
plt.ylabel("$n_{eff}$", fontsize=16)
plt.legend(frameon=False, fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()